# Micro-Transit Eye — corrida para el informe final

Clasificación de ocupación de paraderos: **baja** (0–3 personas) / **media** (4–10) / **alta** (>10).
ResNet-50 vs ViT-B/16, mismo código y mismos datos.

**Qué cambia respecto al notebook del avance** (a partir de la retroalimentación):
- COCO descarta imágenes con anotación `iscrowd` (conteo no confiable cerca de los umbrales).
- `evaluate.py` reporta IC 95% por bootstrap del macro-F1 y mide la latencia con warmup + mediana.
- `baselines.py` compara la clase mayoritaria del train con la mayoritaria de cada split.
- Baseline nuevo con visión: YOLOv8 preentrenado + conteo (`baseline_yolo.py`).
- Interpretabilidad: Grad-CAM (ResNet) y attention rollout (ViT) con `interpret.py`.
- Experimento opcional con Focal Loss.

Correr las celdas **en orden**. Los resultados del avance se respaldan antes de reentrenar (celda de respaldo).

## 1. Setup

In [ ]:
# Montar Drive y clonar/actualizar el repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
if os.path.exists('/content/repo/.git'):
    %cd /content/repo
    !git pull
else:
    !git clone https://github.com/DaniSwi/Micro-Transit-Eye.git /content/repo
    %cd /content/repo

!pip install -q -r requirements.txt

DRIVE = '/content/drive/MyDrive/g7'
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)

# Confirmar que llegaron los arreglos (debe aparecer baseline_yolo.py e interpret.py)
!ls src

In [ ]:
# Respaldar los resultados del avance ANTES de reentrenar: train.py escribe en
# outputs/resnet50 y outputs/vit_b16 y pisaria los checkpoints/metricas ya presentados.
import os
if not os.path.exists(f'{DRIVE}/outputs_avance'):
    !cp -r {DRIVE}/outputs {DRIVE}/outputs_avance
    print('respaldo creado en', f'{DRIVE}/outputs_avance')
else:
    print('ya existe el respaldo, no se toca:', f'{DRIVE}/outputs_avance')

## 2. Datos

### COCO (con el filtro `iscrowd` nuevo)
Se borra el `manifest.csv` viejo porque `write_manifest()` ahora **acumula** entradas: si no se borra,
quedarían mezcladas las imágenes del avance (con conteo sucio) y las nuevas. Las imágenes que ya están
en disco no se vuelven a bajar; solo se descargan las que faltan.

In [ ]:
!rm -f {DRIVE}/data/raw/coco/manifest.csv

!python src/download_selective.py coco \
    --out {DRIVE}/data/raw/coco \
    --cap-per-class 1500 \
    --max-side 640
# Debe aparecer la linea 'N descartadas por tener anotacion iscrowd'

### CrowdHuman
Su filtro no cambió, así que se reutiliza lo extraído en el avance. Orden de búsqueda:
1. `g7/data/raw/crowdhuman/manifest.csv` ya existe → no hace nada.
2. Si no, lo recupera desde el respaldo `g7/data/dataset_g7.tar`.
3. Si tampoco está el respaldo, lo extrae de los ZIP (requiere acceso directo a la carpeta `CrowdHuman` en "Mi unidad").

In [ ]:
import os
CH = '/content/drive/MyDrive/CrowdHuman'
TAR = f'{DRIVE}/data/dataset_g7.tar'
MANIFEST = f'{DRIVE}/data/raw/crowdhuman/manifest.csv'

if not os.path.exists(MANIFEST) and os.path.exists(TAR):
    print('recuperando crowdhuman desde', TAR)
    !tar -xf {TAR} -C {DRIVE}/data raw/crowdhuman

if os.path.exists(MANIFEST):
    !wc -l {MANIFEST}
    print('OK: CrowdHuman listo')
else:
    assert os.path.exists(f'{CH}/annotation_train.odgt'), \
        f'No hay manifest, ni respaldo .tar, ni {CH}: agregar acceso directo a la carpeta CrowdHuman en Mi unidad'
    !mkdir -p {DRIVE}/data/raw/crowdhuman/
    !cp -n "{CH}/annotation_train.odgt" "{DRIVE}/data/raw/crowdhuman/"
    for z in ['CrowdHuman_train01.zip', 'CrowdHuman_train02.zip']:
        if os.path.exists(f'{CH}/{z}'):
            !python src/download_selective.py crowdhuman-extract \
                --zip "{CH}/{z}" \
                --odgt "{DRIVE}/data/raw/crowdhuman/annotation_train.odgt" \
                --out {DRIVE}/data/raw/crowdhuman \
                --cap-per-class 800 \
                --max-side 640
        else:
            print('no esta', z, '- se omite')

### ShanghaiTech Part B
En el avance la extracción decía "OK" pero `labels.py` no lo encontraba: el zip de Kaggle no deja la
carpeta con el nombre exacto que espera el código (`part_B/train_data/ground-truth`). Esta celda busca
los `GT_*.mat` de Part B / train donde sea que hayan quedado y los copia a la ruta esperada.

In [ ]:
import os, glob, shutil

BASE = f'{DRIVE}/data/raw/shanghaitech'
TARGET = f'{BASE}/part_B/train_data'
ZIP_SHANGHAI = '/content/drive/MyDrive/CrowdHuman/archive.zip'
TAR = f'{DRIVE}/data/dataset_g7.tar'

def listo():
    return bool(glob.glob(f'{TARGET}/ground-truth/GT_*.mat')) and bool(glob.glob(f'{TARGET}/images/*.jpg'))

if listo():
    print('OK: ShanghaiTech ya esta en la ruta que espera labels.py')
else:
    if not glob.glob(f'{BASE}/**/GT_*.mat', recursive=True) and os.path.exists(TAR):
        print('recuperando shanghaitech desde', TAR)
        !tar -xf {TAR} -C {DRIVE}/data raw/shanghaitech
    if not glob.glob(f'{BASE}/**/GT_*.mat', recursive=True):
        assert os.path.exists(ZIP_SHANGHAI), \
            f'No hay respaldo .tar ni {ZIP_SHANGHAI}: agregar acceso directo a la carpeta CrowdHuman en Mi unidad'
        print('descomprimiendo', ZIP_SHANGHAI)
        os.makedirs(BASE, exist_ok=True)
        !unzip -n -q "{ZIP_SHANGHAI}" -d "{BASE}"

    mats = [p for p in glob.glob(f'{BASE}/**/GT_*.mat', recursive=True)
            if 'part_b' in p.lower() and 'train' in p.lower()]
    assert mats, 'No se encontraron GT_*.mat de Part B / train. Revisar el contenido del zip con: !unzip -l ZIP | head -40'

    gt_src = os.path.dirname(mats[0])
    parent = os.path.dirname(gt_src)
    img_src = next(os.path.join(parent, d) for d in os.listdir(parent)
                   if glob.glob(os.path.join(parent, d, '*.jpg')))
    print('ground truth encontrado en:', gt_src)
    print('imagenes encontradas en:   ', img_src)

    os.makedirs(TARGET, exist_ok=True)
    if os.path.abspath(gt_src) != os.path.abspath(f'{TARGET}/ground-truth'):
        shutil.copytree(gt_src, f'{TARGET}/ground-truth', dirs_exist_ok=True)
    if os.path.abspath(img_src) != os.path.abspath(f'{TARGET}/images'):
        shutil.copytree(img_src, f'{TARGET}/images', dirs_exist_ok=True)

    print('listo:', listo())

print(len(glob.glob(f'{TARGET}/ground-truth/GT_*.mat')), 'archivos .mat,',
      len(glob.glob(f'{TARGET}/images/*.jpg')), 'imagenes')

In [ ]:
# Verificar las tres fuentes
!echo "--- coco ---"         && du -sh {DRIVE}/data/raw/coco 2>/dev/null
!echo "--- crowdhuman ---"   && du -sh {DRIVE}/data/raw/crowdhuman 2>/dev/null
!echo "--- shanghaitech ---" && du -sh {DRIVE}/data/raw/shanghaitech 2>/dev/null

### Enlaces a Drive
`data/raw/` y `outputs/` del repo clonado vienen vacíos (las imágenes y checkpoints nunca van a git).
Se reemplazan por enlaces a Drive para que el código encuentre los datos y los resultados sobrevivan si se corta la sesión.
Si se reinicia el runtime, hay que volver a correr esta celda.

In [ ]:
!rm -rf /content/repo/data/raw /content/repo/outputs
!ln -sfn {DRIVE}/data/raw /content/repo/data/raw
!ln -sfn {DRIVE}/outputs  /content/repo/outputs
!ls data/raw

### Fotos locales de Valparaíso
Están en la carpeta compartida `aprendizaje_automatico_2`, repartidas en `alto/bajo/medio`. **Antes de correr:**
agregarle un acceso directo en "Mi unidad" (clic derecho → Organizar → Agregar acceso directo), para que aparezca en
`/content/drive/MyDrive/aprendizaje_automatico_2`.

Solo se copian las fotos **`_orig`**. Las variantes `_hflip`, `_bright_inc` y `_bright_dec` son copias de la misma
escena: si entraran a la evaluación inflarían `n` (108 → 432) sin agregar información, y el IC 95% saldría
engañosamente angosto. El `labels.csv` del repo ya trae solo las `_orig`.

In [ ]:
import shutil
from pathlib import Path

FOTOS_VALPO = '/content/drive/MyDrive/aprendizaje_automatico_2'
DEST = Path('data/local_valpo')
DEST.mkdir(parents=True, exist_ok=True)

copiadas = 0
for sub in ['alto', 'bajo', 'medio']:
    origen = Path(FOTOS_VALPO) / sub
    if not origen.exists():
        print('aviso: no existe', origen, '- falta el acceso directo en Mi unidad?')
        continue
    for f in origen.iterdir():
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png') and '_orig.' in f.name:
            shutil.copy(f, DEST / f.name)
            copiadas += 1
print(copiadas, 'fotos _orig copiadas a', DEST)

# Fotos sin fila en labels.csv (o filas sin foto) quedan fuera o rompen el entrenamiento: avisar
import pandas as pd
csv = pd.read_csv('data/local_valpo/labels.csv')
en_disco = {p.name for p in DEST.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')}
en_csv = set(csv['filename'])
print('en labels.csv pero sin foto:', sorted(en_csv - en_disco) or 'ninguna')
print('foto sin fila en labels.csv:', sorted(en_disco - en_csv) or 'ninguna')

## 3. Índice, splits y baselines

In [ ]:
!python src/labels.py --raw-dir data/raw --out data/index.csv
# Ahora deberia aparecer [shanghaitech] N imagenes, no 'no encontrado'

In [ ]:
!python src/splits.py --index data/index.csv --seed 42

In [ ]:
# Imprime la mayoritaria del train Y la mayoritaria de cada split
!python src/baselines.py --index data/index.csv

In [ ]:
# Baseline con vision, sin entrenar nada propio: YOLOv8 + conteo de personas
!python src/baseline_yolo.py --index data/index.csv --split test
!python src/baseline_yolo.py --index data/index.csv --split local

## 4. Entrenamiento

Prueba de humo con 1 época primero (copias temporales de los YAML, no se tocan los del repo).
Si termina sin errores, recién ahí el entrenamiento real. El dataset cambió (filtro `iscrowd` + ShanghaiTech),
así que **los dos modelos se reentrenan desde cero**.

In [ ]:
import yaml, os
os.makedirs('/content/smoketest', exist_ok=True)
for name in ['resnet50', 'vit_b16']:
    cfg = yaml.safe_load(open(f'configs/{name}.yaml'))
    cfg['optim']['epochs'] = 1
    cfg['train']['out_dir'] = f'/content/smoketest/{name}'
    yaml.safe_dump(cfg, open(f'/content/smoketest/{name}.yaml', 'w'))

!python src/train.py --config /content/smoketest/resnet50.yaml
!python src/train.py --config /content/smoketest/vit_b16.yaml

In [ ]:
!python src/train.py --config configs/resnet50.yaml

In [ ]:
!python src/train.py --config configs/vit_b16.yaml

## 5. Evaluación (ahora con IC 95% y latencia corregida)

In [ ]:
!python src/evaluate.py --run outputs/resnet50 --split test
!python src/evaluate.py --run outputs/vit_b16  --split test
!python src/evaluate.py --run outputs/resnet50 --split local
!python src/evaluate.py --run outputs/vit_b16  --split local

## 6. Interpretabilidad
Grad-CAM para ResNet y attention rollout para ViT. Guarda un PNG por imagen en `outputs/<modelo>/interpret_<split>/`,
con la clase real, la predicha y si acertó. Sirve para la lámina de casos de acierto y error.

In [ ]:
!python src/interpret.py --run outputs/resnet50 --split local --n 9
!python src/interpret.py --run outputs/vit_b16  --split local --n 9

In [ ]:
# Ver las figuras aqui mismo
from IPython.display import Image, display
import glob
for modelo in ['resnet50', 'vit_b16']:
    print('=====', modelo, '=====')
    for p in sorted(glob.glob(f'outputs/{modelo}/interpret_local/*.png')):
        display(Image(p, width=600))

## 7. (Opcional) Experimento con Focal Loss
Alternativa a `class_weights` para el desbalance. Se aplica **igual a los dos modelos** (invariante 1)
usando copias temporales de los YAML, y guarda en `outputs/<modelo>_focal` para no pisar la corrida principal.

In [ ]:
import yaml
for name in ['resnet50', 'vit_b16']:
    cfg = yaml.safe_load(open(f'configs/{name}.yaml'))
    cfg['run_name'] = f'{name}_focal'
    cfg['optim']['loss'] = 'focal'
    cfg['optim']['focal_gamma'] = 2.0
    cfg['train']['out_dir'] = f'outputs/{name}_focal'
    yaml.safe_dump(cfg, open(f'/content/{name}_focal.yaml', 'w'))

!python src/train.py --config /content/resnet50_focal.yaml
!python src/train.py --config /content/vit_b16_focal.yaml

for run in ['resnet50_focal', 'vit_b16_focal']:
    for split in ['test', 'local']:
        !python src/evaluate.py --run outputs/{run} --split {split}

## 8. Tabla resumen para el informe

In [ ]:
import json, glob
import pandas as pd

rows = []
for p in sorted(glob.glob('outputs/*/eval_*/metrics.json')):
    m = json.load(open(p))
    rows.append({
        'modelo': m['run'], 'split': m['split'], 'n': m['n_images'],
        'macro_f1': round(m['macro_f1'], 4), 'IC95%': m.get('macro_f1_ci95'),
        'accuracy': round(m['accuracy'], 4),
        'recall_alta': round(m['recall_por_clase']['alta'], 3),
        'ms/imagen': m['ms_por_imagen'], 'params (M)': m['n_params_millions'],
    })
print('MODELOS')
display(pd.DataFrame(rows))

print('BASELINES TRIVIALES')
display(pd.read_csv('outputs/baselines_triviales.csv'))

print('BASELINE YOLO')
display(pd.read_csv('outputs/baseline_yolo/summary.csv'))

print('MACRO-F1 POR FUENTE (riesgo 1: atajo por dataset)')
for p in sorted(glob.glob('outputs/*/eval_test/per_source.csv')):
    print(p)
    display(pd.read_csv(p))